In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SOURCE_TABLE = "retail_demo.bronze.sales_transactions"

SILVER_TABLE = "retail_demo.silver.fact_sales"

QUARANTINE_TABLE = "retail_demo.silver.sales_quarantine"

In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS retail_demo.silver
""")

In [0]:
df_bronze = spark.table(SOURCE_TABLE)

print("Bronze record count:", df_bronze.count())

df_bronze.printSchema()

In [0]:
df_standardized = (
    df_bronze

    .withColumn(
        "transaction_id",
        F.trim(F.col("transaction_id"))
    )

    .withColumn(
        "order_id",
        F.trim(F.col("order_id"))
    )

    .withColumn(
        "order_line_id",
        F.trim(F.col("order_line_id"))
    )

    .withColumn(
        "customer_id",
        F.trim(F.col("customer_id"))
    )

    .withColumn(
        "product_id",
        F.trim(F.col("product_id"))
    )

    .withColumn(
        "store_id",
        F.trim(F.col("store_id"))
    )

    .withColumn(
        "campaign_id",
        F.trim(F.col("campaign_id"))
    )

    .withColumn(
        "source_system",
        F.upper(F.trim(F.col("source_system")))
    )

    .withColumn(
        "currency",
        F.upper(F.trim(F.col("currency")))
    )

    .withColumn(
        "order_status",
        F.upper(F.trim(F.col("order_status")))
    )

    .withColumn(
        "sales_channel",
        F.initcap(F.trim(F.col("sales_channel")))
    )

    .withColumn(
        "payment_method",
        F.initcap(F.trim(F.col("payment_method")))
    )
)


In [0]:
df_quality = (
    df_standardized

    .withColumn(
        "dq_missing_transaction_id",
        F.col("transaction_id").isNull()
    )

    .withColumn(
        "dq_missing_customer_id",
        F.col("customer_id").isNull()
    )

    .withColumn(
        "dq_missing_product_id",
        F.col("product_id").isNull()
    )

    .withColumn(
        "dq_invalid_quantity",
        F.col("quantity").isNull() |
        (F.col("quantity") <= 0)
    )

    .withColumn(
        "dq_invalid_unit_price",
        F.col("unit_price").isNull() |
        (F.col("unit_price") <= 0)
    )

    .withColumn(
        "dq_invalid_discount",
        F.col("discount_pct").isNull() |
        (F.col("discount_pct") < 0) |
        (F.col("discount_pct") > 1)
    )

    .withColumn(
        "dq_invalid_tax",
        F.col("tax_pct").isNull() |
        (F.col("tax_pct") < 0) |
        (F.col("tax_pct") > 1)
    )

    .withColumn(
        "dq_invalid_order_timestamp",
        F.col("order_timestamp").isNull()
    )
)

In [0]:
df_quality = (
    df_quality

    .withColumn(
        "dq_status",

        F.when(
            F.col("dq_missing_transaction_id") |
            F.col("dq_missing_customer_id") |
            F.col("dq_missing_product_id") |
            F.col("dq_invalid_quantity") |
            F.col("dq_invalid_unit_price") |
            F.col("dq_invalid_discount") |
            F.col("dq_invalid_tax") |
            F.col("dq_invalid_order_timestamp"),

            F.lit("INVALID")
        )

        .otherwise(
            F.lit("VALID")
        )
    )
)

In [0]:
df_quality = (
    df_quality

    .withColumn(
        "dq_error_reason",

        F.concat_ws(
            " | ",

            F.when(
                F.col("dq_missing_transaction_id"),
                F.lit("Missing transaction ID")
            ),

            F.when(
                F.col("dq_missing_customer_id"),
                F.lit("Missing customer ID")
            ),

            F.when(
                F.col("dq_missing_product_id"),
                F.lit("Missing product ID")
            ),

            F.when(
                F.col("dq_invalid_quantity"),
                F.lit("Invalid quantity")
            ),

            F.when(
                F.col("dq_invalid_unit_price"),
                F.lit("Invalid unit price")
            ),

            F.when(
                F.col("dq_invalid_discount"),
                F.lit("Invalid discount")
            ),

            F.when(
                F.col("dq_invalid_tax"),
                F.lit("Invalid tax percentage")
            ),

            F.when(
                F.col("dq_invalid_order_timestamp"),
                F.lit("Invalid order timestamp")
            )
        )
    )
)

In [0]:
df_valid = (
    df_quality
    .filter(
        F.col("dq_status") == "VALID"
    )
)

In [0]:
df_quarantine = (
    df_quality
    .filter(
        F.col("dq_status") == "INVALID"
    )
)

In [0]:
dedup_window = (
    Window
    .partitionBy("transaction_id")
    .orderBy(
        F.col("ingestion_timestamp").desc()
    )
)

In [0]:
df_dedup = (
    df_valid

    .withColumn(
        "_row_number",
        F.row_number().over(dedup_window)
    )

    .filter(
        F.col("_row_number") == 1
    )

    .drop("_row_number")
)

In [0]:
df_calculated = (
    df_dedup

    .withColumn(
        "gross_sales",
        F.round(
            F.col("quantity") *
            F.col("unit_price"),
            2
        )
    )

    .withColumn(
        "discount_amount",
        F.round(
            F.col("gross_sales") *
            F.col("discount_pct"),
            2
        )
    )

    .withColumn(
        "net_sales",
        F.round(
            F.col("gross_sales") -
            F.col("discount_amount"),
            2
        )
    )

    .withColumn(
        "tax_amount",
        F.round(
            F.col("net_sales") *
            F.col("tax_pct"),
            2
        )
    )

    .withColumn(
        "order_value",
        F.round(
            F.col("net_sales") +
            F.col("tax_amount") +
            F.coalesce(
                F.col("shipping_fee"),
                F.lit(0)
            ),
            2
        )
    )
)

In [0]:
df_calculated = (
    df_calculated

    .withColumn(
        "order_date",
        F.to_date(F.col("order_timestamp"))
    )

    .withColumn(
        "order_year",
        F.year(F.col("order_timestamp"))
    )

    .withColumn(
        "order_month",
        F.month(F.col("order_timestamp"))
    )

    .withColumn(
        "order_month_name",
        F.date_format(
            F.col("order_timestamp"),
            "MMMM"
        )
    )

    .withColumn(
        "order_quarter",
        F.quarter(F.col("order_timestamp"))
    )

    .withColumn(
        "order_day_of_week",
        F.date_format(
            F.col("order_timestamp"),
            "EEEE"
        )
    )
)

In [0]:
df_calculated = (
    df_calculated

    .withColumn(
        "delivery_days",
        F.when(
            F.col("delivery_date").isNotNull(),
            F.datediff(
                F.col("delivery_date"),
                F.col("ship_date")
            )
        )
    )
)

In [0]:
df_customer = (
    spark.table(
        "retail_demo.bronze.dim_customer"
    )

    .select(
        "customer_id",
        "customer_name",
        "gender",
        "age",
        "city",
        "state",
        "region",
        "segment",
        "loyalty_member",
        "loyalty_points"
    )
)

In [0]:
df_silver = (
    df_calculated

    .join(
        df_customer,
        on="customer_id",
        how="left"
    )
)

In [0]:
df_product = (
    spark.table(
        "retail_demo.bronze.dim_product"
    )

    .select(
        "product_id",
        "product_name",
        "category",
        "subcategory",
        "brand",
        "unit_cost",
        "list_price"
    )
)

In [0]:
df_silver = (
    df_silver

    .join(
        df_product,
        on="product_id",
        how="left"
    )
)

In [0]:
df_silver = (
    df_silver

    .withColumn(
        "total_cost",
        F.round(
            F.col("quantity") *
            F.col("unit_cost"),
            2
        )
    )

    .withColumn(
        "gross_profit",
        F.round(
            F.col("net_sales") -
            F.col("total_cost"),
            2
        )
    )

    .withColumn(
        "profit_margin_pct",

        F.when(
            F.col("net_sales") > 0,

            F.round(
                (
                    F.col("gross_profit") /
                    F.col("net_sales")
                ) * 100,
                2
            )
        )
    )
)

In [0]:
df_store = (
    spark.table(
        "retail_demo.bronze.dim_store"
    )

    .select(
        "store_id",

        F.col("store_name"),

        F.col("city")
        .alias("store_city"),

        F.col("state")
        .alias("store_state"),

        F.col("region")
        .alias("store_region"),

        "store_format"
    )
)

In [0]:
df_silver = (
    df_silver

    .join(
        df_store,
        on="store_id",
        how="left"
    )
)

In [0]:
df_campaign = (
    spark.table(
        "retail_demo.bronze.dim_campaign"
    )

    .select(
        "campaign_id",
        "campaign_name",
        "marketing_channel",
        "budget_inr",
        "objective"
    )
)

In [0]:
df_silver = (
    df_silver

    .join(
        df_campaign,
        on="campaign_id",
        how="left"
    )
)

In [0]:
df_silver = (
    df_silver

    .withColumn(
        "_silver_load_timestamp",
        F.current_timestamp()
    )

    .withColumn(
        "_silver_load_date",
        F.current_date()
    )
)

In [0]:
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

print("Silver fact_sales created successfully.")

In [0]:
(
    df_quarantine.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(QUARANTINE_TABLE)
)

print("Silver quarantine table created successfully.")

In [0]:
bronze_count = spark.table(
    SOURCE_TABLE
).count()

silver_count = spark.table(
    SILVER_TABLE
).count()

quarantine_count = spark.table(
    QUARANTINE_TABLE
).count()

print("Bronze records     :", bronze_count)
print("Silver valid rows  :", silver_count)
print("Quarantine records :", quarantine_count)